In [ ]:
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-6-luna"

In [ ]:
agent = client.beta.agents.create(
    model=MODEL,
    instructions="You speak short sentences. Like caveman.",
)

In [ ]:
import base64

data = b"name:Ax School\nCourses:1000\n"

In [ ]:
with client.beta.agents.sessions.create(
    agent_id=agent.id,
    environment={
        "type": "openai_hosted",
        "container_size": "small",
        "packages": {"python": ["PyYAML==6.0.3"]},
        "network": {
            "access": "disabled",
        },
        "setup_commands": [{"command": "mkdir -p outputs"}],
        "files": [
            {
                "type": "inline",
                "path": "/workspace/course.yaml",
                "data": base64.b64encode(data).decode(),
            }
        ],
    },
    input="Tell me the files i have in my /workspace if they are .yaml open them, and print the parsed object. And save it as JSON to /workspace/outputs/course.json then show me the output of that file.",
    stream=True,
) as events:
    for event in events:
        if event.type == "agent.session.created":
            session_id = event.session.id
        elif event.type == "agent.session.turn.output_text.done":
            print(event.text)
        elif (
            event.type == "agent.session.turn.item.done"
            and event.item.type == "command_execution"
        ):
            print("Command: ", event.item.command)
            print("Output: ", event.item.output)
        else:
            print(event.type)

In [88]:
artifacts = client.beta.agents.sessions.artifacts.list(session_id)

for artifact in artifacts:
    artifact_id = artifact.id
    print(artifact.to_dict())
    content = client.beta.agents.sessions.artifacts.content(
        artifact_id, session_id=session_id
    )
    print(content.content)

{'id': 'artifact_2d7edfd717f249d9b62e8499ba98cd55a381f789853f4ce582', 'created_at': 1793093052, 'environment_id': 'ccarenv_b64_Y2NhcmVudl82YWUwNmQ5NzEwZmM4MTkxOWVhYmEwZjgxMTdkMDhlNg', 'object': 'agent.session.artifact', 'path': '/workspace/outputs/course.json', 'session_id': 'sess_05eb664d18463456006ae06d9683dc819396d997403d94354c', 'size_bytes': 30, 'turn_id': 'turn_05eb664d18463456006ae06d9b37f8819385f9574e00a35ab0'}
b'"name:Ax School Courses:1000"\n'
